# Lab 02 - Hồi quy logistic

Chạy notebook từ thư mục gốc `02. Logistic_Regression`.


## Bước 1 - Đọc dữ liệu


In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: đọc bộ dữ liệu sinh viên và xem qua một lượt."""

import pandas as pd

df = pd.read_csv("data/sinh_vien.csv")

print("Kich thuoc bang (so dong, so cot):", df.shape)
print("\nNam dong dau tien:")
print(df.head())
print("\nSo sinh vien theo ket qua:")
print(df["qua_mon"].value_counts())
print("\nTy le qua mon:", round(df["qua_mon"].mean(), 4))
print("\nSo gio on trung binh theo nhom:")
print(df.groupby("qua_mon")["gio_on"].mean().round(2))


Kich thuoc bang (so dong, so cot): (120, 3)

Nam dong dau tien:
   gio_on  diem_giua_ky  qua_mon
0     5.5           3.0        0
1    19.0           6.0        1
2    14.0           7.7        0
3    11.0           3.1        0
4    10.5           5.8        1

So sinh vien theo ket qua:
qua_mon
1    71
0    49
Name: count, dtype: int64

Ty le qua mon: 0.5917

So gio on trung binh theo nhom:
qua_mon
0     8.29
1    19.89
Name: gio_on, dtype: float64


## Bước 2 - Hàm sigmoid


In [2]:
# -*- coding: utf-8 -*-
"""Bước 2: làm quen hàm sigmoid bằng cách tự cài đặt."""

import numpy as np


def sigmoid(z):
    """Ép một số thực bất kỳ về khoảng từ 0 tới 1."""
    return 1 / (1 + np.exp(-z))


print("Bang gia tri cua ham sigmoid")
print("  z    sigmoid(z)")
for z in [-6, -4, -2, -1, 0, 1, 2, 4, 6]:
    print(f"{z:3d}      {sigmoid(z):.4f}")

print("\nKiem tra sigmoid(-z) = 1 - sigmoid(z):")
for z in [1.0, 2.5, 3.7]:
    print(f"z = {z}: {sigmoid(-z):.6f} va {1 - sigmoid(z):.6f}")

w, b = 0.4, -5.0
print(f"\nMo hinh gia dinh: w = {w}, b = {b}")
print("So gio on   z = w*x + b   Xac suat qua mon")
for gio in [5, 10, 12.5, 15, 20, 25]:
    z = w * gio + b
    print(f"{gio:8.1f}   {z:11.2f}   {sigmoid(z):.4f}")


Bang gia tri cua ham sigmoid
  z    sigmoid(z)
 -6      0.0025
 -4      0.0180
 -2      0.1192
 -1      0.2689
  0      0.5000
  1      0.7311
  2      0.8808
  4      0.9820
  6      0.9975

Kiem tra sigmoid(-z) = 1 - sigmoid(z):
z = 1.0: 0.268941 va 0.268941
z = 2.5: 0.075858 va 0.075858
z = 3.7: 0.024127 va 0.024127

Mo hinh gia dinh: w = 0.4, b = -5.0
So gio on   z = w*x + b   Xac suat qua mon
     5.0         -3.00   0.0474
    10.0         -1.00   0.2689
    12.5          0.00   0.5000
    15.0          1.00   0.7311
    20.0          3.00   0.9526
    25.0          5.00   0.9933


## Bước 3 - Khớp mô hình


In [3]:
# -*- coding: utf-8 -*-
"""Bước 3: để scikit-learn tìm w và b từ dữ liệu thật."""

import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on"]]
y = df["qua_mon"]

mo_hinh = LogisticRegression()
mo_hinh.fit(X, y)

w = float(mo_hinh.coef_[0][0])
b = float(mo_hinh.intercept_[0])
print(f"He so goc w = {w:.6f}")
print(f"He so chan b = {b:.6f}")
print(f"So gio on ung voi xac suat 0.5: {-b / w:.2f} gio")

ban_moi = pd.DataFrame({"gio_on": [5.0, 10.0, 13.0, 20.0, 28.0]})
xac_suat = mo_hinh.predict_proba(ban_moi)[:, 1]
nhan = mo_hinh.predict(ban_moi)

print("\nDu doan cho nam ban moi:")
print("So gio on   Xac suat qua   Nhan")
for gio, p, n in zip(ban_moi["gio_on"], xac_suat, nhan):
    print(f"{gio:8.1f}      {p:.4f}       {n}")


He so goc w = 0.392819
He so chan b = -5.064890
So gio on ung voi xac suat 0.5: 12.89 gio

Du doan cho nam ban moi:
So gio on   Xac suat qua   Nhan
     5.0      0.0431       0
    10.0      0.2429       0
    13.0      0.5104       1
    20.0      0.9422       1
    28.0      0.9974       1


## Bước 4 - Đánh giá mô hình


In [4]:
# -*- coding: utf-8 -*-
"""Bước 4: chia dữ liệu rồi chấm điểm bằng bốn thước đo."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh = LogisticRegression().fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("So sinh vien de hoc     :", len(X_train))
print("So sinh vien de kiem tra:", len(X_test))
print("\nMa tran nham lan")
print(f"TN = {tn:2d}   FP = {fp:2d}")
print(f"FN = {fn:2d}   TP = {tp:2d}")
print(f"\nAccuracy  = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall    = {recall_score(y_test, y_pred):.4f}")
print(f"F1        = {f1_score(y_test, y_pred):.4f}")


So sinh vien de hoc     : 90
So sinh vien de kiem tra: 30

Ma tran nham lan
TN =  9   FP =  3
FN =  1   TP = 17

Accuracy  = 0.8667
Precision = 0.8500
Recall    = 0.9444
F1        = 0.8947


## Bước 5 - Đổi ngưỡng


In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: tự áp ngưỡng và xem precision, recall thay đổi."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mo_hinh = LogisticRegression().fit(X_train, y_train)
p = mo_hinh.predict_proba(X_test)[:, 1]

print("Nguong   So ban doan qua   Precision   Recall")
for nguong in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred = (p >= nguong).astype(int)
    pre = precision_score(y_test, y_pred, zero_division=1)
    rec = recall_score(y_test, y_pred, zero_division=0)
    print(f" {nguong:.1f}           {y_pred.sum():2d}          {pre:.4f}     {rec:.4f}")

print("\nNhan xet: nguong tang thi precision co xu huong tang, recall giam.")


Nguong   So ban doan qua   Precision   Recall
 0.2           24          0.7500     1.0000
 0.3           20          0.8500     0.9444
 0.4           20          0.8500     0.9444
 0.5           20          0.8500     0.9444
 0.6           16          1.0000     0.8889
 0.7           15          1.0000     0.8333
 0.8           13          1.0000     0.7222

Nhan xet: nguong tang thi precision co xu huong tang, recall giam.


## Bước 6 - Mô hình hai biến


In [7]:
# -*- coding: utf-8 -*-
"""Bước 6: thêm điểm giữa kỳ làm biến thứ hai."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/sinh_vien.csv")
y = df["qua_mon"]

for ten, cot in [
    ("Mot bien", ["gio_on"]),
    ("Hai bien", ["gio_on", "diem_giua_ky"]),
]:
    X = df[cot]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=17, stratify=y
    )
    mo_hinh = LogisticRegression().fit(X_train, y_train)
    do_chinh_xac = accuracy_score(y_test, mo_hinh.predict(X_test))
    print(f"{ten}: accuracy tren tap kiem tra = {do_chinh_xac:.4f}")

X = df[["gio_on", "diem_giua_ky"]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mo_hinh = LogisticRegression().fit(X_train, y_train)

print("\nHe so cua tung bien:")
for ten_cot, he_so in zip(X.columns, mo_hinh.coef_[0]):
    print(f"{ten_cot:14s} {he_so:+.4f}")
print(f"{'he so chan':14s} {mo_hinh.intercept_[0]:+.4f}")
print("\nCa hai he so deu duong: on nhieu va diem giua ky cao")
print("deu lam tang xac suat qua mon.")


Mot bien: accuracy tren tap kiem tra = 0.8667
Hai bien: accuracy tren tap kiem tra = 0.9000

He so cua tung bien:
gio_on         +0.3035
diem_giua_ky   +0.5885
he so chan     -6.9811

Ca hai he so deu duong: on nhieu va diem giua ky cao
deu lam tang xac suat qua mon.
